### Let’s train a byte-level BPE tokenizer on the TinyStories dataset

In [11]:
pwd

'/home/vule/projects/cs336/assignment1-basics/nbs'

In [12]:
# create vocabulary
def create_vocab(end_of_text_token, special_tokens):
    """ Create a vocabulary:
    - 256 first bytes
    - end_of_text_token
    - special_tokens
    Input:
        end_of_text_token: str
        special_tokens: list of str
    Output:
        index_2_vocab: dict mapping index to token
    """
    index_2_vocab = {i: bytes([i]) for i in range(256)}
    index_2_vocab[256] = end_of_text_token
    for st in special_tokens:
        index_2_vocab[len(index_2_vocab)] = st
    return index_2_vocab

index_2_vocab = create_vocab('<|endoftext|>', ['<PAD>'])
index_2_vocab[108] + index_2_vocab[111] + index_2_vocab[119]


b'low'

In [13]:
# Parallelizing pre-tokenization

import os
import sys

# Ensure project root (parent of `nbs`) is on sys.path
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))

from cs336_basics.pretokenization_example import find_chunk_boundaries


file_path = "/home/vule/projects/cs336/assignment1-basics/data/TinyStories-sample.txt"
split_special_token = b"<|endoftext|>"
split_special_token_str = "<|endoftext|>"
with open(file_path, "rb") as f:
    num_processes = 1
    boundaries = find_chunk_boundaries(f, num_processes, b"<|endoftext|>")

    for start, end in zip(boundaries[:-1], boundaries[1:]):
        f.seek(start)
        chunk = f.read(end - start).decode("utf-8", errors="ignore")
        # Run pre-tokenization on your chunk and store the counts for each pre-token
        # print(f"--------------------chunk: {chunk}--------------------")
len(chunk)

156696

#### Removing special tokens before pre-tokenization

In [ ]:
import regex as re
import regex as re

def removing_special_tokens(doc, special_tokens):
    """
    Input:
        text: str
        special_tokens: list of str
    Output:
        segments: list of str that are split by special tokens include special tokens
    Example:
        text = "Hello world!<PAD>This is a test<PAD>Bye."
        special_tokens = ["<PAD>"]
        Output: ["Hello world!", "<PAD>", "This is a test", "<PAD>", "Bye."]
    """
    pattern = "|".join(re.escape(tok) for tok in special_tokens)
    # segments = re.split(f"({pattern})", doc)
    segments = re.split(pattern, doc)
    return segments

# test cases
special_tokens = ["<PAD>"]
doc = "Hello world!<PAD>This is a test<PAD>Bye."

true_segments = ["Hello world!", "<PAD>", "This is a test", "<PAD>", "Bye."]
true_segments = ["Hello world!", "This is a test", "Bye."]

segments = removing_special_tokens(doc, special_tokens)

assert true_segments == segments

special_tokens = ["<A|B>", "C+END"]
doc = "Start<A|B>MiddleC+ENDStop"

true_segments = ["Start", "<A|B>", "Middle", "C+END", "Stop"]
true_segments = ["Start", "Middle", "Stop"]

segments = removing_special_tokens(doc, special_tokens)

assert true_segments == segments

special_tokens = ["<PAD>"]
doc = "Hello world!<PAD>This is a test<PAD>Bye."
pattern = "|".join(re.escape(tok) for tok in special_tokens)
re.split(pattern, doc)


['Hello world!', 'This is a test', 'Bye.']

In [15]:
# pre-tokenization
from typing import Any


from collections import Counter
from concurrent.futures import ThreadPoolExecutor

PAT = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""

special_tokens = ["<PAD>"]
text = "Hello world!<PAD>This is a test<PAD>Bye."

def pre_tokenization(chunk, special_tokens):
    """
    Split text using special tokens and PAT
    PAT '(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+
    Input:
        text: str
    Output:
        pre_token_freq: Counter of pre-tokens

    Example:
        text = "Hello world!<PAD>This is a test<PAD>Bye."
        special_tokens = ["<PAD>"]
        Output: Counter({'Hello': 1, 'world': 1, 'This': 1, 'is': 1, 'a': 1, 'test': 1, 'Bye': 1})
    """
    segments = removing_special_tokens(chunk, special_tokens)
    return Counter[Any](sum([re.findall(PAT, segment) for segment in segments], []))

# test cases
special_tokens = ["<PAD>", "<|endoftext|>"]
chunk = "Hello world!<PAD>This is a test<PAD>Bye."
pre_token_freq = pre_tokenization(chunk, special_tokens)
assert pre_token_freq == Counter({'Hello': 1, ' world': 1, '!': 1, 'This': 1, ' is': 1, ' a': 1, ' test': 1, 'Bye': 1, '.': 1})

pre_token_freq = {tuple(k.encode('utf-8')): v for k, v in pre_token_freq.items()}

In [16]:
# create linked list of each pre-token to tracking and update pair
class Node:
    def __init__(self, vocab_index, n_count=0):
        self.vocab_index = vocab_index
        self.next = None
        self.prev = None
        self.n_count = n_count

    def __repr__(self):
        return str(index_2_vocab[self.prev.vocab_index] if self.prev else None) + "-" + str(index_2_vocab[self.vocab_index]) + "-" + str(index_2_vocab[self.next.vocab_index] if self.next else None) + " (" + str(self.n_count) + ")"


def create_linked_list(pre_token_freq):
    freq_linked_list = {}

    for k, v in pre_token_freq.items():
        nodes = tuple([Node(i, v) for i in k])
        for pre, n in zip(nodes, nodes[1:]):
            pre.next = n
            n.prev = pre
        freq_linked_list[nodes] = v

    return freq_linked_list

freq_linked_list = create_linked_list(pre_token_freq)

# freq_linked_list


In [17]:
# create max heap

import heapq

pair_version = {}
pair_count = {}
max_heap = []

def update_pair_count(n1, n2):
    pair_bytes = (index_2_vocab[n1.vocab_index], index_2_vocab[n2.vocab_index])
    if pair_bytes not in pair_count:
        pair_count[pair_bytes] = {'n_count': n1.n_count, 'pair_nodes': [(n1, n2)]}
    else:
        pair_count[pair_bytes]['n_count'] += n1.n_count
        pair_count[pair_bytes]['pair_nodes'].append((n1, n2))

def update_pair_version(n1, n2):
    pair_bytes = (index_2_vocab[n1.vocab_index], index_2_vocab[n2.vocab_index])
    if pair_bytes not in pair_version:
        pair_version[pair_bytes] = 1
    else:
        pair_version[pair_bytes] += 1


def update_max_heap(n1, n2):
    pair_bytes = (index_2_vocab[n1.vocab_index], index_2_vocab[n2.vocab_index])
    heapq.heappush(max_heap, (-pair_count[pair_bytes]['n_count'], -n1.vocab_index, -n2.vocab_index, pair_bytes, pair_version[pair_bytes]))

# init the pair_count

# pair_count is a dict, with key is pair of bytes, and value is the list of pair node.
# pair_version is a dict, with key is pair of bytes, and value is the version of the pair. Used to check out of date pair.
# pair_max_heap is max heap of pair_count, 
def init_max_heap(freq_linked_list):
    for ns, v in freq_linked_list.items():
        for n, next_n in zip(ns, ns[1:]):
            update_pair_count(n, next_n)
            update_pair_version(n, next_n)
            update_max_heap(n, next_n)

pre_token_freq = Counter({'Hello': 1, ' world': 1, '!': 1, 'This': 1, ' is': 1, ' a': 1, ' test': 1, 'Bye': 1, '.': 1})
pre_token_freq = {tuple(k.encode('utf-8')): v for k, v in pre_token_freq.items()}
freq_linked_list = create_linked_list(pre_token_freq)
init_max_heap(freq_linked_list)

assert max_heap[0] == (-2, -105, -115, (b'i', b's'), 2)
    

In [18]:
def remove_pair(n1, n2):
    if n1 and n2:
        pair_bytes = (index_2_vocab[n1.vocab_index], index_2_vocab[n2.vocab_index])
        # update pair count
        # print(f"before remove pair: {pair_bytes} - {pair_count[pair_bytes]}")
        pair_count[pair_bytes]['n_count'] -= n1.n_count
        # print(f"after remove pair: {pair_bytes} - {pair_count[pair_bytes]}")
        # update pair version
        pair_version[pair_bytes] += 1
        # update max heap
        update_max_heap(n1, n2)


def update_max_pair(byte_pair, new_vocab_index):
    list_max_pair = pair_count[byte_pair]
    for n1, n2 in list_max_pair['pair_nodes']:
        remove_pair(n1.prev, n1)
        remove_pair(n2, n2.next)

        # update new node
        new_node = Node(new_vocab_index, n1.n_count)
        new_node.next = n2.next
        if n2.next:
            n2.next.prev = new_node
            update_pair_count(new_node, n2.next); update_pair_version(new_node, n2.next); update_max_heap(new_node, n2.next)
        new_node.prev = n1.prev
        if n1.prev:
            n1.prev.next = new_node
            update_pair_count(n1.prev, new_node); update_pair_version(n1.prev, new_node); update_max_heap(n1.prev, new_node)

def merge():
    while len(max_heap) > 0 and len(merged_pair) < n_vocab:
        # 1. get max pair
        (score, w1_index, w2_index, max_pair, version) = heapq.heappop(max_heap)
        # check pair_version
        if pair_version[max_pair] != version:
            continue
        # 2. update vocab
        new_vocab_index = len(index_2_vocab)
        index_2_vocab[new_vocab_index] = max_pair[0] + max_pair[1]
        # 3. update merge pair
        merged_pair.append((max_pair[0], max_pair[1], max_pair[0] + max_pair[1]))
        # 4. update max pair: update old links and new node
        update_max_pair(max_pair, new_vocab_index)
        


pair_version = {}
pair_count = {}
max_heap = []
merged_pair = []
pre_token_freq = Counter({'Hello': 1, ' world': 1, '!': 1, 'This': 1, ' is': 1, ' a': 1, ' test': 1, 'Bye': 1, '.': 1})
pre_token_freq = {tuple(k.encode('utf-8')): v for k, v in pre_token_freq.items()}
freq_linked_list = create_linked_list(pre_token_freq)
init_max_heap(freq_linked_list)

assert max_heap[0] == (-2, -105, -115, (b'i', b's'), 2)

# merge max pair -> update pair_count -> update max_heap -> update pair_version
n_vocab = 1
merge()
merged_pair == [(b'i', b's', b'is')]


True

In [ ]:
merged_pair

[(b'i', b's', b'is')]

: 

In [ ]:
# Full flow
# Constants
end_of_text_token = "<|endoftext|>"
special_tokens = ["<PAD>"]
file_path = "/home/vule/projects/cs336/assignment1-basics/data/TinyStoriesV2-GPT4-train.txt"
n_vocab = 1000
pair_version = {}
pair_count = {}
max_heap = []
merged_pair = []
# 1. Create vocabulary
vocab_2_index = create_vocab(end_of_text_token, special_tokens)

from collections import Counter
from concurrent.futures import ThreadPoolExecutor


def pre_tokenization_parallel(docs, special_tokens):
    """Batch(32)-parallel pre-tokenization for a list of docs."""
    # pre_tokenization(docs, ...) now batches the list into size-32 and parallelizes batches.
    return pre_tokenization(docs, special_tokens)


# 2. Pre-tokenization
with open(file_path, "rb") as f:
    num_processes = 5
    boundaries = find_chunk_boundaries(f, num_processes, b"<|endoftext|>")
    print(boundaries)
    pre_token_freq = Counter()
    for start, end in zip(boundaries[:-1], boundaries[1:]):
        f.seek(start)
        chunk = f.read(end - start).decode("utf-8", errors="ignore")
        print(f"chunk: {len(chunk)}")
        pre_token_freq += pre_tokenization(chunk, special_tokens)
pre_token_freq = {tuple(k.encode('utf-8')): v for k, v in pre_token_freq.items()}
# create linked list 
freq_linked_list = create_linked_list(pre_token_freq)
init_max_heap(freq_linked_list)
# Train BPE tokenizer
merge()

[0, 445550788, 891101671, 1336652984, 1782202745, 2227753162]
chunk: 445367857
chunk: 445371469
chunk: 445369062
chunk: 445367784
chunk: 445369096
